**Set environment**

In [1]:
import numpy  as np
import pandas as pd
import os

In [2]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



## Import data

**Check file existent**

In [3]:
%%bash -s "{FD_DAT}"
echo $1

/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data


In [4]:
!ls $FD_DAT/variant_bluestarr_richard/closed

annotated_snv_sites_filtered.tsv.gz  regions.fasta.gz
annotated_snv_sites.tsv.gz	     regions_GoF_only.fasta.gz


**Import table**

In [5]:
### set file directory
txt_fdiry = os.path.join(FD_DAT, "variant_bluestarr_richard", "closed")
txt_fname = "annotated_snv_sites_filtered.tsv.gz"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### read table
dat = pd.read_csv(txt_fpath, sep = "\t")

### assign and show
dat_variant_import = dat
print(dat.shape)
dat.head()

(1884977, 6)


,region,pos0,delta,ref,obs,unobs
0,chr4:74487576-74488141,74487586,0.964984,A,G,C
1,chr9:135685425-135685942,135685432,0.945423,G,G,C
2,chr1:173251481-173251541,173251490,0.910127,G,G,C
3,chr1:147523084-147523503,147523391,0.908800,G,G,C
4,chr18:54580694-54581420,54580749,0.862296,A,G,T


## Arrange table

In [6]:
### init
dat = dat_variant_import.copy()

### split region into chrom/start/end
dat_chrom_info    = dat["region"].str.split("[:-]", expand=True)
dat["Chrom"]      = dat_chrom_info[0]
dat["ChromStart"] = dat_chrom_info[1].astype(int)
dat["ChromEnd"]   = dat_chrom_info[2].astype(int)

### rename original columns
dat = dat.rename(columns={
    "region": "Region",
    "pos0":   "Pos0",
    "delta":  "Delta",
    "ref":    "Ref",
    "obs":    "Obs",
    "unobs":  "Unobs"
})

### add Variant_ID
dat["Variant_ID"] = (
    dat["Chrom"].astype(str) + ":" +
    dat["Pos0"].astype(str)  + ":" +
    dat["Ref"].astype(str)   + ":" +
    dat["Obs"].astype(str)   + ":" +
    dat["Unobs"].astype(str)
)

### Reorder columns
dat = dat[[
    "Chrom", "ChromStart", "ChromEnd", "Region", 
    "Variant_ID", 
    "Pos0",
    "Ref", "Obs", "Unobs",
    "Delta"
]]

### sort by Delta (highest -> smallest)
dat = dat.sort_values(by="Delta", ascending=False)

### assign and show
dat_variant_arrange = dat
print(dat.shape)
dat.head()

(1884977, 10)


,Chrom,ChromStart,ChromEnd,Region,Variant_ID,Pos0,Ref,Obs,Unobs,Delta
0,chr4,74487576,74488141,chr4:74487576-74488141,chr4:74487586:A:G:C,74487586,A,G,C,0.964984
1,chr9,135685425,135685942,chr9:135685425-135685942,chr9:135685432:G:G:C,135685432,G,G,C,0.945423
2,chr1,173251481,173251541,chr1:173251481-173251541,chr1:173251490:G:G:C,173251490,G,G,C,0.910127
3,chr1,147523084,147523503,chr1:147523084-147523503,chr1:147523391:G:G:C,147523391,G,G,C,0.908800
4,chr18,54580694,54581420,chr18:54580694-54581420,chr18:54580749:A:G:T,54580749,A,G,T,0.862296


**Export table**

```
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.tsv.gz"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### write table
dat_variant_arrange.to_csv(txt_fpath, sep="\t", index=False, compression="gzip")
```

In [7]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.tsv"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### write table
dat_variant_arrange.to_csv(txt_fpath, sep="\t", index=False)